# Protein interface prediction with graph neural networks

## Plan and research question
Can shared graph encoders improve residue-pair contact prediction over an MLP using the same protein features?

1. Verify Kaggle GPUs and the publisher's DB5-Plus archive checksum.
2. Construct separate 20-neighbor graphs from **unbound** protein chains. Use amino acid, secondary structure, surface, sequence and local geometry features. Never use cross-partner distances.
3. Fit feature preprocessing on 140 official training complexes only. Split the 35 official validation complexes into 18 checkpoint-selection and 17 threshold-calibration complexes, using a fixed seed before examining performance.
4. Train an MLP baseline and three GNN seeds, up to 150 epochs with early stopping. Use the two T4 GPUs for independent training jobs.
5. Freeze checkpoints using validation average precision. Choose each classifier's F1 threshold using calibration pairs only.
6. Evaluate all 55 test complexes on their full residue-pair distribution. Export precision, recall, F1, ROC-AUC, average precision, MCC, confusion counts, per-complex results, predictions and checkpoints.

## References and scope
- [Fout et al., NeurIPS 2017](https://papers.neurips.cc/paper/7231-protein-interface-prediction-using-graph-convolutional-networks.pdf)
- [DIPS-Plus / DB5-Plus data](https://zenodo.org/records/5134732)
- [Publisher's implementation](https://github.com/BioinfoMachineLearning/DIPS-Plus)

This is a modern educational experiment, not an exact reproduction of either paper. This dataset was obtained from its research publisher and is processed **inside Kaggle**; it is not being relabeled as a Kaggle-hosted dataset.

**Prior exposure:** These underlying DB5 test complexes were examined in earlier project experiments. These results are a fixed benchmark rerun, not an independently blind holdout. Exact sequence-pair overlap across splits is checked; family-level homology leakage is not ruled out. Generic 0.75 or 0.90 F1 targets are not guaranteed for sparse residue contacts.


In [ ]:
import os, sys, subprocess, platform, json
from pathlib import Path
import torch
print('KAGGLE CLOUD:', platform.platform())
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2 in Kaggle Settings before running.'
# dill supports the publisher's serialized namedtuple; no legacy DGL dependency is required.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dill==0.4.0'], check=True)
os.environ['GNN_RUN_DIR']='/kaggle/working/interface_run'
RUN=Path(os.environ['GNN_RUN_DIR'])
RUN.mkdir(exist_ok=True)


In [ ]:
# Preserve the entire experiment implementation with the notebook output.
TRAIN_SOURCE='"""Reproducible DB5-Plus interface experiment for Kaggle\'s two T4 GPUs.\n\nCommands: prepare, train --kind {mlp,gnn} --seed N, evaluate.\nNo test labels are used for training, early stopping, scaling or threshold choice.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport tarfile\nimport time\nimport urllib.request\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom scipy.spatial.distance import cdist\nfrom sklearn.metrics import (average_precision_score, roc_auc_score,\n    precision_recall_curve, confusion_matrix, matthews_corrcoef,\n    balanced_accuracy_score, auc, roc_curve)\n\nROOT = Path(os.environ.get(\'GNN_RUN_DIR\', \'/kaggle/working/interface_run\'))\nSOURCE = \'https://zenodo.org/records/5134732/files/final_raw_db5.tar.gz?download=1\'\nMD5 = \'9cbc07672e705f9ba8549168b06e1e06\'\nSEEDS = [224, 225, 226]\nAA = \'ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL UNK\'.split()\nSS = [\'H\', \'B\', \'E\', \'G\', \'I\', \'T\', \'S\', \'-\', \'?\']\nSCALARS = [\'rsa_value\', \'rd_value\', \'avg_cx\', \'s_avg_cx\', \'s_ch_avg_cx\',\n           \'s_ch_s_avg_cx\', \'max_cx\', \'min_cx\', \'cn_value\']\n\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef vector(value, width):\n    try:\n        a = np.asarray(value, dtype=np.float32).ravel()\n    except (TypeError, ValueError):\n        return np.full(width, np.nan, dtype=np.float32)\n    return a if len(a) == width else np.full(width, np.nan, dtype=np.float32)\n\n\ndef graph(frame):\n    df = frame.loc[frame.atom_name == \'CA\'].copy()\n    if not df.index.is_unique:\n        raise ValueError(\'CA dataframe index must be unique\')\n    coords = df[[\'x\', \'y\', \'z\']].to_numpy(dtype=np.float32)\n    if len(df) < 2 or not np.isfinite(coords).all():\n        raise ValueError(\'Invalid protein coordinates\')\n    continuous = np.column_stack([df[SCALARS].to_numpy(dtype=np.float32),\n        np.stack([vector(v, 42) for v in df.hsaac]),\n        np.stack([vector(v, 27) for v in df.sequence_feats])])\n    continuous[~np.isfinite(continuous)] = np.nan\n    res = np.eye(len(AA), dtype=np.float32)[[AA.index(v) if v in AA else len(AA)-1 for v in df.resname]]\n    ss = np.eye(len(SS), dtype=np.float32)[[SS.index(v) if v in SS else len(SS)-1 for v in df.ss_value]]\n    pos = np.arange(len(df), dtype=np.float32)[:, None] / max(len(df)-1, 1)\n    x = np.column_stack([res, ss, pos, continuous])\n    # Intramolecular geometry only. Cross-partner distances would leak the bound pose.\n    distances = cdist(coords, coords).astype(np.float32)\n    np.fill_diagonal(distances, np.inf)\n    k = min(20, len(df)-1)\n    hood = np.argsort(distances, axis=1, kind=\'stable\')[:, :k]\n    d = np.take_along_axis(distances, hood, axis=1)\n    # Scalar distance features preserve rigid rotation/translation invariance.\n    rbf = np.exp(-((d[..., None]-np.linspace(0, 20, 16)) / 2.0)**2)\n    gap = np.minimum(np.abs(np.arange(len(df))[:, None]-hood), 32)[..., None]/32.0\n    edge = np.concatenate([rbf, gap], axis=-1).astype(np.float32)\n    seq = \'-\'.join(df.resname.astype(str))\n    return x, hood.astype(np.int64), edge, df.index.to_numpy(), seq\n\n\ndef prepare():\n    ROOT.mkdir(parents=True, exist_ok=True)\n    archive = ROOT / \'final_raw_db5.tar.gz\'\n    if not archive.exists():\n        print(\'Downloading the official DB5-Plus archive (42.8 MB)\', flush=True)\n        for attempt in range(3):\n            try:\n                urllib.request.urlretrieve(SOURCE, archive)\n                break\n            except Exception:\n                if attempt == 2:\n                    raise\n                time.sleep(3)\n    actual = hashlib.md5(archive.read_bytes()).hexdigest()\n    if actual != MD5:\n        raise ValueError(f\'Archive checksum mismatch: {actual}\')\n    cases = []\n    manifest = {\'source\': SOURCE, \'md5\': actual, \'label_policy\':\n        \'Released positive CA row-index pairs; full CA Cartesian complement is negative, \'\n        \'following DIPS-Plus build_complex_labels. Stored sampled neg_idx is not used.\',\n        \'splits\': {}, \'cases\': [], \'prior_test_exposure\':\n        \'Underlying DB5 test complexes were examined in earlier project experiments. \'\n        \'This is a fixed benchmark rerun, not a new blind holdout.\'}\n    with tarfile.open(archive) as tar:\n        for split in (\'train\', \'val\', \'test\'):\n            names = tar.extractfile(f\'raw/pairs-postprocessed-{split}.txt\').read().decode().splitlines()\n            manifest[\'splits\'][split] = []\n            for name in names:\n                # Pickle is used only for this checksum-verified, publisher-provided archive.\n                obj = pd.read_pickle(io.BytesIO(tar.extractfile(\'raw/\'+name).read()))\n                gl, gr = graph(obj.df0), graph(obj.df1)\n                xl, hl, el, il, sl = gl\n                xr, hr, er, ir, sr = gr\n                if not all(\'_u_\' in str(s) for s in obj.srcs.values()):\n                    raise ValueError(f\'{name}: expected unbound input structures\')\n                left, right = {v:i for i,v in enumerate(il)}, {v:i for i,v in enumerate(ir)}\n                positive = np.asarray(obj.pos_idx, dtype=np.int64)\n                if positive.ndim != 2 or positive.shape[1] != 2:\n                    raise ValueError(f\'{name}: invalid contact labels\')\n                try:\n                    pos = np.array([left[a]*len(xr)+right[b] for a,b in positive], dtype=np.int64)\n                except KeyError as exc:\n                    raise ValueError(f\'{name}: contact refers to a missing CA atom\') from exc\n                if len(np.unique(pos)) != len(pos) or not 0 < len(pos) < len(xl)*len(xr):\n                    raise ValueError(f\'{name}: duplicate or degenerate contact labels\')\n                # Check the released sampled negatives against the exhaustive complement.\n                neg = np.array([left[a]*len(xr)+right[b] for a,b in obj.neg_idx], dtype=np.int64)\n                if np.intersect1d(pos, neg).size:\n                    raise ValueError(f\'{name}: conflicting positive/negative labels\')\n                code = Path(name).stem\n                seq_key = hashlib.sha256(\'|\'.join(sorted([sl, sr])).encode()).hexdigest()\n                cases.append(dict(code=code, split=split, xl=xl, xr=xr, hl=hl, hr=hr,\n                                  el=el, er=er, pos=pos, sequence_pair_hash=seq_key))\n                manifest[\'splits\'][split].append(code)\n                manifest[\'cases\'].append(dict(code=code, split=split, n_left=len(xl),\n                    n_right=len(xr), n_pairs=len(xl)*len(xr), positives=len(pos),\n                    prevalence=len(pos)/(len(xl)*len(xr)), sequence_pair_hash=seq_key))\n                if len(cases) % 25 == 0:\n                    print(f\'Checked {len(cases)}/230 complexes\', flush=True)\n    if [len(manifest[\'splits\'][s]) for s in (\'train\',\'val\',\'test\')] != [140,35,55]:\n        raise ValueError(\'Unexpected official split sizes\')\n    # Exact pair-sequence duplicates across splits would be a concrete leakage error.\n    seen = {}\n    for c in cases:\n        key = c[\'sequence_pair_hash\']\n        if key in seen and seen[key][\'split\'] != c[\'split\']:\n            raise ValueError(f"Duplicate sequence pair across splits: {seen[key][\'code\']}, {c[\'code\']}")\n        seen[key] = c\n    # Hold back part of the official validation set for threshold fitting only.\n    val = np.array(manifest[\'splits\'][\'val\'])\n    perm = np.random.default_rng(224).permutation(len(val))\n    manifest[\'selection_ids\'] = val[perm[:18]].tolist()\n    manifest[\'calibration_ids\'] = val[perm[18:]].tolist()\n    groups = [set(manifest[\'splits\'][\'train\']), set(manifest[\'selection_ids\']),\n              set(manifest[\'calibration_ids\']), set(manifest[\'splits\'][\'test\'])]\n    assert all(not a & b for i,a in enumerate(groups) for b in groups[i+1:])\n    # Fit imputation and feature scaling on the training proteins only.\n    fit = np.concatenate([c[k] for c in cases if c[\'split\']==\'train\' for k in (\'xl\',\'xr\')])\n    median = np.nanmedian(fit, axis=0)\n    median = np.nan_to_num(median).astype(np.float32)\n    filled = np.where(np.isfinite(fit), fit, median)\n    mean, std = filled.mean(0), filled.std(0)\n    std = np.maximum(std, 0.05)\n    np.savez_compressed(ROOT/\'feature_scaler.npz\', median=median, mean=mean, std=std)\n    data_dir = ROOT/\'processed\'\n    data_dir.mkdir(exist_ok=True)\n    for c in cases:\n        for key in (\'xl\',\'xr\'):\n            x = c[key]\n            missing = (~np.isfinite(x)).astype(np.float32)\n            scaled = np.clip((np.where(np.isfinite(x),x,median)-mean)/std, -10, 10)\n            c[key] = np.column_stack([scaled,missing]).astype(np.float32)\n            assert np.isfinite(c[key]).all()\n        np.savez_compressed(data_dir/f"{c[\'code\']}.npz", **{k:c[k] for k in\n            (\'xl\',\'xr\',\'hl\',\'hr\',\'el\',\'er\',\'pos\')})\n    manifest[\'feature_dimension\'] = int(cases[0][\'xl\'].shape[1])\n    manifest[\'edge_dimension\'] = int(cases[0][\'el\'].shape[-1])\n    manifest[\'normalization\'] = \'Training-only median imputation and z scaling, clipped at +/-10; missing masks appended.\'\n    manifest[\'sequence_check_limit\'] = \'Exact pair sequences checked; shared chains and homologous families can still cross splits.\'\n    write_json(ROOT/\'manifest.json\', manifest)\n    print(\'DATA READY:\', {s:len(v) for s,v in manifest[\'splits\'].items()},\n          \'selection=18, calibration=17, feature dimension=\', manifest[\'feature_dimension\'], flush=True)\n\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, width, edges, neighbors):\n        super().__init__()\n        self.neighbors = neighbors\n        self.center = nn.Linear(width, width)\n        if neighbors:\n            self.message = nn.Linear(width, width, bias=False)\n            self.gate = nn.Sequential(nn.Linear(edges, 32), nn.SiLU(), nn.Linear(32,1), nn.Sigmoid())\n        self.norm = nn.LayerNorm(width)\n        self.dropout = nn.Dropout(0.25)\n\n    def forward(self, x, hood, edge):\n        update = self.center(x)\n        if self.neighbors:\n            weights = self.gate(edge)\n            pooled = (x[hood]*weights).sum(1)/weights.sum(1).clamp_min(1e-6)\n            update = update+self.message(pooled)\n        return self.norm(x+self.dropout(nn.functional.silu(update)))\n\n\nclass Model(nn.Module):\n    def __init__(self, dim, edges, neighbors=True, width=128):\n        super().__init__()\n        self.project = nn.Sequential(nn.Linear(dim,width), nn.LayerNorm(width), nn.SiLU())\n        self.layers = nn.ModuleList([ResidualBlock(width,edges,neighbors) for _ in range(3)])\n        self.pair = nn.Sequential(nn.Linear(width*3,width), nn.SiLU(), nn.Dropout(0.25),\n                                  nn.Linear(width,64), nn.SiLU(), nn.Linear(64,1))\n        self.interface = nn.Linear(width,1)\n\n    def encode(self,x,h,e):\n        x = self.project(x)\n        for layer in self.layers:\n            x = layer(x,h,e)\n        return x\n\n    def embed(self,c):\n        return self.encode(c[\'xl\'],c[\'hl\'],c[\'el\']), self.encode(c[\'xr\'],c[\'hr\'],c[\'er\'])\n\n    def score(self,l,r,flat):\n        a,b = l[flat//len(r)],r[flat%len(r)]\n        return self.pair(torch.cat([a+b,torch.abs(a-b),a*b],-1)).flatten()\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.set_num_threads(2)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef load_case(code, device):\n    with np.load(ROOT/\'processed\'/f\'{code}.npz\', allow_pickle=False) as z:\n        c = {k:torch.from_numpy(z[k]).to(device) for k in z.files}\n    n = len(c[\'xl\'])*len(c[\'xr\'])\n    c[\'y\'] = np.zeros(n, dtype=np.uint8)\n    c[\'y\'][c[\'pos\'].cpu().numpy()] = 1\n    c[\'positive\'] = np.flatnonzero(c[\'y\'])\n    c[\'negative\'] = np.flatnonzero(1-c[\'y\'])\n    c[\'code\'] = code\n    c[\'node_l\'] = torch.zeros(len(c[\'xl\']),device=device)\n    c[\'node_r\'] = torch.zeros(len(c[\'xr\']),device=device)\n    c[\'node_l\'][c[\'pos\']//len(c[\'xr\'])] = 1\n    c[\'node_r\'][c[\'pos\']%len(c[\'xr\'])] = 1\n    return c\n\n\n@torch.no_grad()\ndef predict(model,c,chunk=8192):\n    model.eval()\n    l,r = model.embed(c)\n    scores=[]\n    for start in range(0,len(c[\'y\']),chunk):\n        idx = torch.arange(start,min(start+chunk,len(c[\'y\'])),device=l.device)\n        scores.append(model.score(l,r,idx).cpu().numpy())\n    out=np.concatenate(scores)\n    assert np.isfinite(out).all()\n    return out\n\n\ndef train(kind, seed, epochs, patience):\n    if not torch.cuda.is_available() and os.environ.get(\'GNN_ALLOW_CPU\') != \'1\':\n        raise RuntimeError(\'Kaggle GPU unavailable; do not silently start the long run on CPU\')\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    seed_all(seed)\n    manifest=json.loads((ROOT/\'manifest.json\').read_text())\n    training=[load_case(c,device) for c in manifest[\'splits\'][\'train\']]\n    validation=[load_case(c,device) for c in manifest[\'selection_ids\']]\n    model=Model(manifest[\'feature_dimension\'],manifest[\'edge_dimension\'],kind==\'gnn\').to(device)\n    opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-3)\n    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode=\'max\',factor=0.5,patience=6)\n    rng=np.random.default_rng(seed)\n    folder=ROOT/f\'{kind}_{seed}\'\n    folder.mkdir(exist_ok=True)\n    best,wait,history=-1.,0,[]\n    print(\'TRAINING\',kind,seed,\'on\',torch.cuda.get_device_name(0) if device.type==\'cuda\' else \'CPU\',\n          \'parameters\',sum(p.numel() for p in model.parameters()),flush=True)\n    for epoch in range(1,epochs+1):\n        start=time.time()\n        model.train()\n        total=0.\n        for i in rng.permutation(len(training)):\n            c=training[int(i)]\n            # Resample negatives each epoch. Balanced TRAINING only, never validation/test.\n            p=rng.choice(c[\'positive\'],size=min(512,len(c[\'positive\'])),replace=False)\n            n=rng.choice(c[\'negative\'],size=min(len(c[\'negative\']),10*len(p)),replace=False)\n            idx=np.concatenate([p,n])\n            target=torch.from_numpy(c[\'y\'][idx].astype(np.float32)).to(device)\n            idx=torch.from_numpy(idx).to(device)\n            opt.zero_grad(set_to_none=True)\n            l,r=model.embed(c)\n            score=model.score(l,r,idx)\n            # Effective balanced pair objective; logits are NOT calibrated probabilities.\n            loss=nn.functional.binary_cross_entropy_with_logits(score,target,\n                    pos_weight=score.new_tensor(len(n)/len(p)))\n            aux=(nn.functional.binary_cross_entropy_with_logits(model.interface(l).flatten(),c[\'node_l\'])+\n                 nn.functional.binary_cross_entropy_with_logits(model.interface(r).flatten(),c[\'node_r\']))/2\n            loss=loss+0.2*aux\n            if not torch.isfinite(loss):\n                raise ValueError(\'Non-finite training loss\')\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(),5.)\n            opt.step()\n            total+=float(loss.detach())\n        pred=[predict(model,c) for c in validation]\n        ap=float(np.mean([average_precision_score(c[\'y\'],s) for c,s in zip(validation,pred)]))\n        auc_median=float(np.median([roc_auc_score(c[\'y\'],s) for c,s in zip(validation,pred)]))\n        scheduler.step(ap)\n        row=dict(epoch=epoch,loss=total/len(training),selection_macro_ap=ap,\n                 selection_median_auc=auc_median,lr=opt.param_groups[0][\'lr\'],seconds=time.time()-start)\n        history.append(row)\n        if ap>best+1e-5:\n            best,wait=ap,0\n            torch.save({\'state\':model.state_dict(),\'kind\':kind,\'seed\':seed,\'epoch\':epoch,\n                \'selection_macro_ap\':ap,\'dim\':manifest[\'feature_dimension\'],\n                \'edges\':manifest[\'edge_dimension\']},folder/\'best.pt\')\n        else:\n            wait+=1\n        pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\n        print(f\'{kind} seed={seed} epoch={epoch:03d} loss={row["loss"]:.4f} \'\n              f\'validation AP={ap:.5f} median AUC={auc_median:.4f} \'\n              f\'best AP={best:.5f} wait={wait}/{patience} seconds={row["seconds"]:.1f}\',flush=True)\n        if wait>=patience:\n            break\n    write_json(folder/\'completed.json\',dict(kind=kind,seed=seed,epochs=len(history),best_selection_ap=best))\n\n\ndef threshold_from_calibration(y,score):\n    # Compute exact F1 over distinct score thresholds, including score ties correctly.\n    p,r,t=precision_recall_curve(y,score)\n    f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-15)\n    # If tied, choose the higher threshold (fewer predicted positives).\n    i=int(np.flatnonzero(np.isclose(f,f.max(),rtol=0,atol=1e-12))[-1])\n    return float(t[i]),float(f[i])\n\n\ndef measures(y,score,threshold):\n    pred=score>=threshold\n    tn,fp,fn,tp=map(int,confusion_matrix(y,pred,labels=[0,1]).ravel())\n    precision=tp/max(tp+fp,1)\n    recall=tp/max(tp+fn,1)\n    p,r,_=precision_recall_curve(y,score)\n    return dict(n_pairs=len(y),positives=int(y.sum()),prevalence=float(y.mean()),\n        roc_auc=float(roc_auc_score(y,score)),average_precision=float(average_precision_score(y,score)),\n        trapezoid_pr_auc=float(auc(r,p)),precision=precision,recall=recall,\n        f1=2*tp/max(2*tp+fp+fn,1),mcc=float(matthews_corrcoef(y,pred)),\n        accuracy=(tp+tn)/len(y),balanced_accuracy=float(balanced_accuracy_score(y,pred)),\n        specificity=tn/max(tn+fp,1),tp=tp,fp=fp,fn=fn,tn=tn,threshold=threshold)\n\n\ndef top_precision(y,score,k):\n    k=min(k,len(y))\n    boundary=np.partition(score,len(score)-k)[len(score)-k]\n    above=score>boundary\n    tied=score==boundary\n    tp=float(y[above].sum())+(k-int(above.sum()))*float(y[tied].mean())\n    return tp/k\n\n\ndef evaluate():\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    manifest=json.loads((ROOT/\'manifest.json\').read_text())\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    calibration=[load_case(c,device) for c in manifest[\'calibration_ids\']]\n    test=[load_case(c,device) for c in manifest[\'splits\'][\'test\']]\n    calibration_y=np.concatenate([c[\'y\'] for c in calibration])\n    test_y=np.concatenate([c[\'y\'] for c in test])\n    models={}\n    selected={}\n    for kind,seed in [(\'mlp\',224)]+[(\'gnn\',s) for s in SEEDS]:\n        ckpt=torch.load(ROOT/f\'{kind}_{seed}\'/\'best.pt\',map_location=device,weights_only=True)\n        model=Model(ckpt[\'dim\'],ckpt[\'edges\'],kind==\'gnn\').to(device)\n        model.load_state_dict(ckpt[\'state\'])\n        name=f\'{kind}_{seed}\'\n        # Architecture sanity checks on a calibration example only.\n        c=calibration[0]\n        model.eval()\n        with torch.no_grad():\n            l,r=model.embed(c)\n            flat=torch.tensor([0,len(c[\'y\'])//2,len(c[\'y\'])-1],device=device)\n            swap=(flat%len(r))*len(l)+(flat//len(r))\n            torch.testing.assert_close(model.score(l,r,flat),model.score(r,l,swap),atol=1e-5,rtol=1e-5)\n        models[name]={\'calibration\':[predict(model,c) for c in calibration],\n                      \'test\':[predict(model,c) for c in test]}\n        selected[name]={\'epoch\':ckpt[\'epoch\'],\'selection_macro_ap\':ckpt[\'selection_macro_ap\']}\n        del model\n    # Predeclared equal logit ensemble; no seed or model is selected on test scores.\n    models[\'gnn_ensemble\']={split:[np.mean([models[f\'gnn_{s}\'][split][i] for s in SEEDS],axis=0)\n        for i in range(len(calibration) if split==\'calibration\' else len(test))]\n        for split in (\'calibration\',\'test\')}\n    summaries={}\n    records=[]\n    outputs=ROOT/\'predictions\'\n    outputs.mkdir(exist_ok=True)\n    fig,axes=plt.subplots(1,3,figsize=(16,4.5))\n    for name,scores in models.items():\n        cp=np.concatenate(scores[\'calibration\'])\n        threshold,cal_f1=threshold_from_calibration(calibration_y,cp)\n        ts=np.concatenate(scores[\'test\'])\n        summary=measures(test_y,ts,threshold)\n        summary[\'calibration_f1\']=cal_f1\n        summary[\'calibration_pairs\']=len(calibration_y)\n        per=[]\n        for c,s in zip(test,scores[\'test\']):\n            row=measures(c[\'y\'],s,threshold)\n            row.update(model=name,complex=c[\'code\'],precision_at_10=top_precision(c[\'y\'],s,10),\n                precision_at_l=top_precision(c[\'y\'],s,min(len(c[\'xl\']),len(c[\'xr\']))))\n            per.append(row)\n            records.append(row)\n            np.savez_compressed(outputs/f"{name}_{c[\'code\']}.npz",y=c[\'y\'],logit=s,threshold=threshold)\n        summary.update(median_complex_roc_auc=float(np.median([x[\'roc_auc\'] for x in per])),\n            macro_average_precision=float(np.mean([x[\'average_precision\'] for x in per])),\n            macro_f1=float(np.mean([x[\'f1\'] for x in per])),\n            mean_precision_at_10=float(np.mean([x[\'precision_at_10\'] for x in per])))\n        # Resample whole complexes for intervals, preserving within-complex dependence.\n        rng=np.random.default_rng(224)\n        hashes={c[\'code\']:c[\'sequence_pair_hash\'] for c in manifest[\'cases\']}\n        grouped={}\n        for row in per:\n            key=hashes[row[\'complex\']]\n            grouped.setdefault(key,np.zeros(3,dtype=np.int64))\n            grouped[key]+=np.array([row[\'tp\'],row[\'fp\'],row[\'fn\']])\n        counts=np.stack(list(grouped.values()))\n        boot=[]\n        for _ in range(2000):\n            tp,fp,fn=counts[rng.integers(0,len(counts),len(counts))].sum(0)\n            boot.append(2*tp/max(2*tp+fp+fn,1))\n        summary[\'pooled_f1_complex_bootstrap_95ci\']=np.quantile(boot,[0.025,0.975]).tolist()\n        summary[\'bootstrap_sequence_pair_groups\']=len(counts)\n        summaries[name]=summary\n        if name in (\'mlp_224\',\'gnn_ensemble\'):\n            p,r,_=precision_recall_curve(test_y,ts)\n            axes[0].plot(r,p,label=name)\n            fpr,tpr,_=roc_curve(test_y,ts)\n            axes[1].plot(fpr,tpr,label=name)\n            axes[2].bar(np.arange(3)+(0 if name==\'mlp_224\' else .36),\n                [summary[\'precision\'],summary[\'recall\'],summary[\'f1\']],.36,label=name)\n        print(\'RESULT\',name,json.dumps(summary),flush=True)\n    axes[0].axhline(test_y.mean(),color=\'gray\',ls=\'--\',label=\'contact prevalence\')\n    axes[0].set(xlabel=\'Recall\',ylabel=\'Precision\',title=\'Full-candidate test precision-recall\')\n    axes[1].plot([0,1],[0,1],\'k--\',alpha=.5)\n    axes[1].set(xlabel=\'False positive rate\',ylabel=\'True positive rate\',title=\'Test ROC\')\n    axes[2].set_xticks(np.arange(3)+.18,[\'Precision\',\'Recall\',\'F1\'])\n    axes[2].set(title=\'Threshold chosen on calibration only\',ylim=(0,1))\n    for a in axes:\n        a.legend()\n    fig.tight_layout()\n    fig.savefig(ROOT/\'test_results.png\',dpi=180)\n    plt.close(fig)\n    pd.DataFrame(records).to_csv(ROOT/\'per_complex_metrics.csv\',index=False)\n    pd.DataFrame(summaries).T.to_csv(ROOT/\'summary_metrics.csv\')\n    pd.DataFrame([{\'model\':n,**v} for n,v in selected.items()]).to_csv(ROOT/\'selected_checkpoints.csv\',index=False)\n    report={\'models\':summaries,\'selected_checkpoints\':selected,\'runtime\':{\n        \'platform\':platform.platform(),\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\n        \'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},\n        \'protocol\':{\'training_complexes\':140,\'selection_complexes\':18,\'threshold_calibration_complexes\':17,\n            \'test_complexes\':55,\'max_epochs\':150,\'early_stopping_patience\':20,\n            \'checkpoint_metric\':\'mean per-complex validation average precision\',\n            \'threshold\':\'pooled F1 maximum on 17 calibration complexes only\',\n            \'heldout_distribution\':\'Every CA residue pair, with no negative downsampling\',\n            \'main_output\':\'Pair contact prediction, not auxiliary single-residue interface classification\',\n            \'prior_test_exposure\':manifest[\'prior_test_exposure\'],\n            \'uncertainty\':\'Bootstrap resamples exact sequence-pair groups, conditional on fitted models and threshold; \'\n            \'3 GNN seeds report training variation. Homology dependencies are not removed.\'}}\n    write_json(ROOT/\'results.json\',report)\n    (ROOT/\'COMPLETE.txt\').write_text(\'Completed successfully. Read results.json and manifest.json.\\n\')\n    print(\'EXPERIMENT COMPLETE. Artifacts:\',ROOT,flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'command\',choices=[\'prepare\',\'train\',\'evaluate\'])\n    parser.add_argument(\'--kind\',choices=[\'mlp\',\'gnn\'],default=\'gnn\')\n    parser.add_argument(\'--seed\',type=int,default=224)\n    parser.add_argument(\'--epochs\',type=int,default=150)\n    parser.add_argument(\'--patience\',type=int,default=20)\n    args=parser.parse_args()\n    if args.command==\'prepare\': prepare()\n    elif args.command==\'train\': train(args.kind,args.seed,args.epochs,args.patience)\n    else: evaluate()\n'
Path('/kaggle/working/train.py').write_text(TRAIN_SOURCE, encoding='utf-8')
print('Saved train.py:',len(TRAIN_SOURCE),'characters')


In [ ]:
# Verify source data, build graphs, and freeze the split before training.
subprocess.run([sys.executable, '-u', '/kaggle/working/train.py', 'prepare'], check=True)
manifest=json.loads((RUN/'manifest.json').read_text())
print('Split sizes:', {k:len(v) for k,v in manifest['splits'].items()})
print('Checkpoint selection:',len(manifest['selection_ids']),
      'Threshold calibration:',len(manifest['calibration_ids']))


## Training objective and model selection

The shared GNN encoder applies three residual neighborhood layers. Distance-based edge gates weight messages from neighbors within each chain. A symmetric pair head uses the sum, absolute difference and product of the two residue embeddings. A small auxiliary interface loss trains the encoder; the reported primary task remains **residue-pair contact prediction**.

Training resamples negatives at 10 per positive and balances their loss contribution. All validation, calibration and test pairs remain exhaustive. As a result, training logits are **not calibrated probabilities**. A default 0.5 sigmoid cutoff would be unjustified. We choose the logit cutoff on the separate calibration complexes instead.

The stopping rule maximizes mean per-complex validation average precision. The test labels are never consulted by the training processes. Each process saves its best checkpoint and epoch history. The MLP shares encoder width, pair head and auxiliary objective; removing neighborhood messages changes the parameter count, so it is a feature/width-matched baseline rather than an exact parameter-matched comparison.


In [ ]:
import time
# Independent jobs use separate GPUs. No test-based model or seed selection.
waves=[ [('mlp',224,0),('gnn',224,1)], [('gnn',225,0),('gnn',226,1)] ]
for wave in waves:
    active=[]
    for kind,seed,gpu in wave:
        label=f'{kind}_{seed}'
        log_path=RUN/f'{label}.log'
        handle=log_path.open('w', buffering=1)
        env=os.environ.copy()
        env['CUDA_VISIBLE_DEVICES']=str(gpu)
        env['PYTHONUNBUFFERED']='1'
        p=subprocess.Popen([sys.executable,'-u','/kaggle/working/train.py','train',
            '--kind',kind,'--seed',str(seed),'--epochs','150','--patience','20'],
            env=env,stdout=handle,stderr=subprocess.STDOUT)
        active.append((label,p,handle,log_path))
        print('STARTED',label,'Kaggle GPU',gpu,'pid',p.pid,flush=True)
    last={}
    while any(p.poll() is None for _,p,_,_ in active):
        for label,p,handle,path in active:
            lines=path.read_text(errors='replace').splitlines()
            line=lines[-1] if lines else 'starting'
            if last.get(label)!=line:
                print(label,':',line,flush=True)
                last[label]=line
        time.sleep(20)
    failures=[]
    for label,p,handle,path in active:
        handle.close()
        print(label,'exit',p.returncode,flush=True)
        if p.returncode:
            failures.append(label)
            print(path.read_text(errors='replace')[-12000:],flush=True)
    if failures:
        raise RuntimeError('Training failed: '+', '.join(failures))
print('ALL FOUR TRAINING RUNS FINISHED',flush=True)


## Evaluation and interpretation

- **Precision** = TP / (TP + FP): how many proposed contacts are real?
- **Recall** = TP / (TP + FN): how many real contacts were recovered?
- **F1** = 2 TP / (2 TP + FP + FN): their harmonic balance at the calibration-selected threshold.
- **Average precision** summarizes contact ranking in the precision-recall curve. It is distinct from trapezoidal PR area. Compare it with the printed contact prevalence.
- **ROC-AUC** measures overall ranking; a high AUC can coexist with low precision for rare contacts.
- **MCC** and **balanced accuracy** include both classes. Raw accuracy is accompanied by an always-negative reference.
- **P@10** is computed with fractional credit for ties at the cutoff.

Scores are reported both pooled across pairs and by complex. A sequence-pair-group bootstrap provides a conditional interval for pooled F1. Three GNN runs show variation across training seeds. These do not establish performance on a homology-disjoint external benchmark.

The equal-weight GNN logit ensemble is predeclared. Individual seeds and the ensemble are all reported, regardless of which test result looks best. The test scores are measured outcomes and may remain modest. Larger training duration alone does not guarantee improvement.


In [ ]:
subprocess.run([sys.executable,'-u','/kaggle/working/train.py','evaluate'],check=True)
import pandas as pd
from IPython.display import display, Image
results=json.loads((RUN/'results.json').read_text())
table=pd.DataFrame(results['models']).T
display(table[['precision','recall','f1','roc_auc','average_precision','mcc',
               'median_complex_roc_auc','mean_precision_at_10','tp','fp','fn','tn']])
prevalence=float(table.iloc[0]['prevalence'])
print('Always-negative accuracy:',1-prevalence,'; recall=0, F1=0, MCC=0')
display(Image(filename=str(RUN/'test_results.png')))
print('All checkpoints, histories, predictions and metrics:',RUN)
assert (RUN/'COMPLETE.txt').exists()


In [ ]:
# Export a compact results package plus the separately available prediction files.
import zipfile
bundle=Path('/kaggle/working/Protein_Interface_GNN_Kaggle_Results.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN.rglob('*'):
        if p.is_file() and 'processed' not in p.parts and p.suffix!='.gz':
            z.write(p,p.relative_to(RUN))
    z.write('/kaggle/working/train.py','train.py')
print('RESULTS READY:',bundle,'bytes:',bundle.stat().st_size)
